In [111]:
!pip install -q nltk gensim gdown

In [112]:
import os
import gzip
import nltk
import numpy as np

from google.colab import drive
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec, FastText, KeyedVectors

In [113]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [114]:
from google.colab import drive
drive.mount('/content/drive')
input_file = "/content/drive/My Drive/nlp/exp1/cleaned_output.txt"
output_file = "/content/drive/My Drive/nlp/exp3/embedded_output.txt"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [115]:
with open(input_file, "r", encoding="utf-8") as file:
    text = file.read()

print("Input Text\n")
print(text)

Input Text

hello everyone name param im currently learning natural language processing nlp usually study hours every day yesterday studied hours exam th july questions feel free email backup email also visit blog check github page weather today amazing scored last assignment completed coding exercises solved programming questions semester machine learning deep learning data science python programming artificial intelligence cloud computing cybersecurity favorite subjects quick brown fox jumps lazy dog cat sleeps old wooden table meanwhile group students discussing algorithms databases operating systems computer networks library thank reading sample text wonderful day


In [116]:
sentences_text = nltk.sent_tokenize(text)

sentences = [
    [word.lower() for word in word_tokenize(sentence) if word.isalnum()]
    for sentence in sentences_text
]

# Flatten into one list of words
tokens = [word for sentence in sentences for word in sentence]

In [117]:
print("\nTraining Word2Vec...\n")

word2vec_model = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    epochs=100
)


Training Word2Vec...



In [118]:
if not os.path.exists("glove-twitter-25.gz"):
    !gdown https://github.com/RaRe-Technologies/gensim-data/releases/download/glove-twitter-25/glove-twitter-25.gz

print("Loading GloVe Model...\n")

word_vectors = {}

with gzip.open("glove-twitter-25.gz", "rt", encoding="utf-8") as f:
    for line in f:
        values = line.strip().split()
        if len(values) == 26:
            word = values[0]
            vector = np.array(values[1:], dtype="float32")
            word_vectors[word] = vector

glove_model = KeyedVectors(vector_size=25)

glove_model.add_vectors(
    list(word_vectors.keys()),
    list(word_vectors.values())
)

print("GloVe Loaded Successfully\n")

Loading GloVe Model...

GloVe Loaded Successfully



In [119]:
test_word = None

for word in tokens:
    if (
        word in word2vec_model.wv
        and word in fasttext_model.wv
        and word in glove_model
    ):
        test_word = word
        break

if test_word is None:
    raise ValueError("No common word found in all three models.")

print("Test Word :", test_word)

Test Word : hello


In [120]:
# Word2Vec Results
vector = word2vec_model.wv[test_word]
similar = word2vec_model.wv.most_similar(test_word, topn=5)

# FastText Results
ft_vector = fasttext_model.wv[test_word]
ft_similar = fasttext_model.wv.most_similar(test_word, topn=5)

# GloVe Results
glove_vector = glove_model[test_word]
glove_similar = glove_model.most_similar(test_word, topn=5)

# Cosine Similarity
word1 = tokens[0]
word2 = tokens[1]
similarity = word2vec_model.wv.similarity(word1, word2)

In [121]:
print("\nWord2Vec Similar Words\n")
for word, score in similar:
    print(f"{word:<15} {score:.4f}")

print("\nFastText Similar Words\n")
for word, score in ft_similar:
    print(f"{word:<15} {score:.4f}")

print("\nGloVe Similar Words\n")
for word, score in glove_similar:
    print(f"{word:<15} {score:.4f}")

print("\nCosine Similarity")
print(word1, "-", word2)
print(similarity)


Word2Vec Similar Words

usually         0.4237
intelligence    0.3503
email           0.3407
databases       0.3397
github          0.3342

FastText Similar Words

helps           0.9996
program         0.9996
intelligence    0.9996
artificial      0.9996
interpret       0.9996

GloVe Similar Words

thanks          0.9293
thank           0.9241
birthday        0.9219
welcome         0.9199
happy           0.9158

Cosine Similarity
hello - everyone
0.1373978


In [122]:
with open(output_file, "w", encoding="utf-8") as f:
    f.write("Input Text\n\n")
    f.write(text)
    f.write("\n\n")

    f.write("Test Word\n\n")
    f.write(test_word)
    f.write("\n\n")

    f.write("Word2Vec\n\n")
    f.write("Vector\n\n")
    f.write(str(vector))
    f.write("\n\n")

    f.write("Most Similar Words\n\n")
    for word, score in similar:
        f.write(f"{word:<15} {score:.4f}\n")
    f.write("\n\n")

    f.write("FastText\n\n")
    f.write("Vector\n\n")
    f.write(str(ft_vector))
    f.write("\n\n")

    f.write("Most Similar Words\n\n")
    for word, score in ft_similar:
        f.write(f"{word:<15} {score:.4f}\n")
    f.write("\n\n")

    f.write("GloVe\n\n")
    f.write("Vector\n\n")
    f.write(str(glove_vector))
    f.write("\n\n")

    f.write("Most Similar Words\n\n")
    for word, score in glove_similar:
        f.write(f"{word:<15} {score:.4f}\n")
    f.write("\n\n")

    f.write("Cosine Similarity\n\n")
    f.write(f"Word 1 : {word1}\n")
    f.write(f"Word 2 : {word2}\n")
    f.write(f"Similarity : {similarity:.4f}\n")

print("\nOutput saved successfully.")
print(output_file)


Output saved successfully.
/content/drive/My Drive/nlp/exp3/embedded_output.txt
